# Reddit Topic Modeling API Notebook

This notebook demonstrates core pipeline components such as text cleaning, fastText embedding, KMeans clustering, and t-SNE visualization.

In [ ]:
# ==============================================================
# Reddit Topic Modeling API Notebook
# ==============================================================
# This notebook demonstrates core pipeline components such as
# text cleaning, fastText embedding, K-Means clustering, and
# t-SNE visualization using helper functions from reddit_utils.py
# ==============================================================

# Example imports
import fasttext
import pandas as pd
import numpy as np
from reddit_utils import clean_text, average_vector
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import nltk

nltk.download('stopwords')

# -----------------------------------------
# 1️⃣  Text Cleaning Example
# -----------------------------------------
sample_comment = "Breaking: Government announces new climate policy! Visit https://news.com/article for details."
cleaned_comment = clean_text(sample_comment)

print("Original Comment:\n", sample_comment)
print("\nCleaned Comment:\n", cleaned_comment)


# -----------------------------------------
# 2️⃣  Generating Word Embeddings
# -----------------------------------------
# Load the pretrained fastText model (English 300-dim).
# The model file (~4 GB) can be downloaded once and reused.
model_path = "cc.en.300.bin"

# Uncomment the two lines below the first time you run:
# !wget -O cc.en.300.bin.gz https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz
# !gunzip cc.en.300.bin.gz

model = fasttext.load_model(model_path)

# Compute average embedding for cleaned text
vector = average_vector(model, cleaned_comment)

print("\nEmbedding shape:", vector.shape)
print("Embedding preview (first 10 dims):", vector[:10])


# -----------------------------------------
# 3️⃣  Mini Clustering + Visualization Demo
# -----------------------------------------
comments = [
    "The government passed a new law on climate change.",
    "The stock market fell sharply after the election.",
    "Citizens protested for better education policies.",
    "A new technology startup raised millions in funding.",
    "Sports fans celebrated the championship victory!"
]

cleaned_comments = [clean_text(c) for c in comments]
vectors = np.vstack([average_vector(model, c) for c in cleaned_comments])

# Apply K-Means clustering
kmeans = KMeans(n_clusters=2, random_state=42)
labels = kmeans.fit_predict(vectors)

# Reduce to 2D using t-SNE for visualization
tsne = TSNE(n_components=2, perplexity=5, random_state=42)
reduced = tsne.fit_transform(vectors)

# Plot clusters
plt.figure(figsize=(6,4))
plt.scatter(reduced[:,0], reduced[:,1], c=labels, cmap='viridis', s=60)
for i, txt in enumerate(labels):
    plt.text(reduced[i,0]+0.5, reduced[i,1], f"Topic {txt}", fontsize=9)
plt.title("Mini Demo: K-Means + t-SNE on Sample Comments")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.show()
